# Model Design
----

## Review
- Datasets, Dataloaders and Transforms
- We mightt want to perform additional transformation for a more sensible features
  - feature transformation to get `home_size=total_rooms/households`, `home_density=total_bedrooms/total_rooms`,`household_size=population/households`

In [2]:
import torch
from torch import nn
import pandas as pd

from torch.utils.data import Dataset, DataLoader

from pathlib import Path

### Feature Engineering
- Feature Transformation
- Normalization
- Feature Selection

In [3]:
from sklearn.preprocessing import StandardScaler
import os
import numpy as np

In [4]:
  def feature_transform(df):
    # this internal method will serve as transform method for both the features and atarges
    df['home_size'] = df['total_rooms']/df['households']
    df['home_density'] = df['total_bedrooms']/df['total_rooms']
    df['household_size'] = df['population']/df['households']
    return df[['longitude','latitude','housing_median_age','median_income','home_size','home_density','household_size','median_house_value']]

In [5]:
dataset_path = Path("/content/sample_data")

train_df = pd.read_csv(dataset_path/"california_housing_train.csv")
test_df = pd.read_csv(dataset_path/"california_housing_test.csv")

# Feature Transform
train_df = feature_transform(train_df)
test_df = feature_transform(test_df)

print(test_df.columns)


# Normalization (we accept outliers for now)
## Fit scaler to train only to avoid data leakage
scaler = StandardScaler()
scaler.fit(train_df)

train_np = scaler.transform(train_df)
test_np = scaler.transform(test_df)

# Save as preprocessed data
PREPROCESSED_DIR = Path("/content/preprocessed_data")
os.makedirs(name=PREPROCESSED_DIR, exist_ok=True)
np.savez_compressed(PREPROCESSED_DIR/"train.npz", data=train_np, )
np.savez_compressed(PREPROCESSED_DIR/"test.npz", data=test_np)


Index(['longitude', 'latitude', 'housing_median_age', 'median_income',
       'home_size', 'home_density', 'household_size', 'median_house_value'],
      dtype='object')


In [6]:
# test load
# with np.load(PREPROCESSED_DIR/"train.npz") as data:
#     X = data['data']
#     print(X[:, :8].shape)

In [7]:
## Dataset

def to_tensor(x):
  return torch.from_numpy(x)

class HousingDataset(Dataset):
  def __init__(self, root: Path, train: bool = True, transform=None, target_transform=None):
    with np.load(PREPROCESSED_DIR/f"{'train' if train else 'test' }.npz") as data:
        X,y = data['data'][:,:7], data['data'][:,7:8]

    if transform:
      self.X = transform(X)
    if target_transform:
      self.y = transform(y)


  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx,], self.y[idx,]


train_dataset = HousingDataset(dataset_path, train=True, transform=to_tensor, target_transform=to_tensor)
test_dataset = HousingDataset(dataset_path, train=False, transform=to_tensor, target_transform=to_tensor)

### Hyperparameters
- batch_size=32
- epochs=8
- learning_rate=1e-2

In [8]:
batch_size=32
epochs=8
learning_rate=1e-2

In [9]:
# for idx, (X, y) in enumerate(test_dataset):
#   print(idx)
#   print(X.shape, X)
#   print(y.shape, y)
#   break

In [10]:
train_dataloader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

In [11]:
for X,y in train_dataloader:
  print(X.shape, y.shape)
  break

torch.Size([32, 7]) torch.Size([32, 1])


## Building a Neural Network
Here we want to build a `Regression` model that identifies the `median house value` of a house based on final features.

In [12]:
class Regressor(nn.Module):
  def __init__(self, in_features, out_features, hidden_features):
    # 3 layer dense network
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(in_features=in_features, out_features=hidden_features),
        nn.ReLU(),
        nn.Linear(hidden_features, hidden_features//2),
        nn.ReLU(),
        nn.Linear(hidden_features//2, hidden_features//2),
        nn.ReLU(),
        nn.Linear(hidden_features//2, out_features)
    )

  def forward(self, X):
    return self.layers(X)

model = Regressor(7, 1, 16)

In [13]:
for params in model.parameters():
  print(params.shape)

torch.Size([16, 7])
torch.Size([16])
torch.Size([8, 16])
torch.Size([8])
torch.Size([8, 8])
torch.Size([8])
torch.Size([1, 8])
torch.Size([1])


In [14]:
# random inference
X=torch.rand(5,7)
model(X)

tensor([[0.3209],
        [0.3262],
        [0.3221],
        [0.3174],
        [0.3205]], grad_fn=<AddmmBackward0>)

## Training
- we use the MSE loss function since this is a regression problem.

In [15]:
# define the loss function
loss_fn = nn.MSELoss()
# define the optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [20]:
# the training loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def train_loop(dataloader, model, loss_fn, optimizer):
  model.train()
  size = len(dataloader.dataset)
  for batch, (X,y) in enumerate(dataloader):
    # print(f"Epoch {epoch} | Batch {batch}")

    # Cast input tensors to float32
    X,y = X.to(device).float(), y.to(device).float()
    # print(X.dtype, y.dtype)

    pred = model(X)
    loss = loss_fn(pred, y)

    optimizer.zero_grad() #reset grad
    loss.backward() # compute gradient,
    optimizer.step() #update params

    # print(f"Loss: {loss.item()}")
    if batch % 100 == 0:
        loss, current = loss.item(), batch * batch_size + len(X)
        print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
  model.eval()
  with torch.no_grad():
    X_test = dataloader.dataset.X.to(device).float()
    y_test = dataloader.dataset.y.to(device).float()
    pred = model(X_test)
    loss = loss_fn(pred, y_test)
    print(f"Validation Loss for epoch {epoch}: {loss.item()}")
    pass

In [22]:
for epoch in range(epochs):
  print(f"Epoch {epoch}")
  train_loop(train_dataloader, model, loss_fn, optimizer)
  test_loop(test_dataloader, model, loss_fn)

Epoch 0
loss: 0.149300  [   32/17000]
loss: 0.376436  [ 3232/17000]
loss: 0.299869  [ 6432/17000]
loss: 0.164238  [ 9632/17000]
loss: 0.159645  [12832/17000]
loss: 0.253181  [16032/17000]
Validation Loss for epoch 0: 0.27428510785102844
Epoch 1
loss: 0.332320  [   32/17000]
loss: 0.223831  [ 3232/17000]
loss: 0.164183  [ 6432/17000]
loss: 0.219725  [ 9632/17000]
loss: 0.327006  [12832/17000]
loss: 0.463448  [16032/17000]
Validation Loss for epoch 1: 0.2794724106788635
Epoch 2
loss: 0.143982  [   32/17000]
loss: 0.216746  [ 3232/17000]
loss: 0.351109  [ 6432/17000]
loss: 0.467703  [ 9632/17000]
loss: 0.254235  [12832/17000]
loss: 0.350848  [16032/17000]
Validation Loss for epoch 2: 0.26987364888191223
Epoch 3
loss: 0.164216  [   32/17000]
loss: 0.257474  [ 3232/17000]
loss: 0.147227  [ 6432/17000]
loss: 0.253246  [ 9632/17000]
loss: 0.202802  [12832/17000]
loss: 0.262283  [16032/17000]
Validation Loss for epoch 3: 0.2755541205406189
Epoch 4
loss: 0.121870  [   32/17000]
loss: 0.397430  

## Save and Load the Model


In [24]:
torch.save(model.state_dict(), "regressor.pth")

In [25]:
model.load_state_dict(torch.load("regressor.pth"))

<All keys matched successfully>

##